# 🗂️ Notebook 2: Airbnb — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/airbnb
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **User** — host or guest.
- **Listing** — a property; owned by a host.
- **Availability** — per-day state for each listing (free/blocked/booked).
- **Booking** — a range reserved by a guest, links to payment.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from datetime import date
from decimal import Decimal
from typing import Literal
from pydantic import BaseModel, Field

class Listing(BaseModel):
    id: int
    host_id: int
    title: str
    lat: float
    lng: float
    price_per_night: Decimal
    max_guests: int = Field(ge=1)

class Booking(BaseModel):
    id: int
    listing_id: int
    guest_id: int
    check_in: date
    check_out: date
    status: Literal["pending", "confirmed", "cancelled"] = "pending"

    def nights(self) -> int:
        return (self.check_out - self.check_in).days

l = Listing(id=1, host_id=42, title="Cabin", lat=47.6, lng=-122.3,
            price_per_night=Decimal("120"), max_guests=4)
b = Booking(id=1, listing_id=1, guest_id=7,
            check_in=date(2026,5,1), check_out=date(2026,5,4))
print(l)
print(b, "nights=", b.nights())

## HTTP APIs

| Method | Path | What |
|---|---|---|
| GET | `/search?lat=..&lng=..&in=..&out=..` | Search available listings |
| GET | `/listings/{id}` | Listing detail + calendar |
| POST | `/bookings` | Create booking (server checks availability atomically) |
| DELETE | `/bookings/{id}` | Cancel booking |


## Quick demo

In [ ]:
# Toy in-memory booking check
bookings = []  # list of (listing_id, check_in, check_out)

def overlaps(a_in, a_out, b_in, b_out):
    return a_in < b_out and b_in < a_out

def book(listing_id, check_in, check_out):
    for lid, ci, co in bookings:
        if lid == listing_id and overlaps(ci, co, check_in, check_out):
            raise ValueError("dates not available")
    bookings.append((listing_id, check_in, check_out))
    return "confirmed"

from datetime import date
print(book(1, date(2026,5,1), date(2026,5,4)))
try:
    book(1, date(2026,5,3), date(2026,5,6))
except ValueError as e:
    print("rejected:", e)

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.